# MLflow RAGAS Scores Query

This notebook fetches `ragas_scores.json` artifacts from MLflow runs and computes mean scores grouped by:
- `question_class`
- `subdomain`
- (`question_class`, `subdomain`)

In [1]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Config
MLFLOW_TRACKING_URI = "http://localhost:8567"
EXPERIMENT_NAME =  "langchain-rag-20260416" #None  # Set to a name string to filter one experiment
MAX_RUNS = 200

# Use one of: "latest", "all", or a specific run_id
RUN_SELECTION = "latest"

RAGAS_TABLE_ARTIFACT = "ragas_scores.json"
METRIC_COLS = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "answer_relevance",
    "factual_correctness",
]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: http://localhost:8567


In [3]:
def list_artifacts_recursive(run_id: str, path: str = ""):
    items = []
    for art in client.list_artifacts(run_id, path):
        items.append(art.path)
        if art.is_dir:
            items.extend(list_artifacts_recursive(run_id, art.path))
    return items


def normalize_logged_table(json_obj) -> pd.DataFrame:
    # mlflow.log_table can be stored in different JSON shapes depending on version.
    if isinstance(json_obj, list):
        return pd.DataFrame(json_obj)

    if isinstance(json_obj, dict):
        if "data" in json_obj and "columns" in json_obj:
            return pd.DataFrame(json_obj["data"], columns=json_obj["columns"])
        return pd.DataFrame(json_obj)

    raise ValueError("Unsupported ragas_scores.json structure")


def load_ragas_table_for_run(run_id: str):
    artifact_paths = list_artifacts_recursive(run_id)

    # Find ragas_scores.json at any artifact depth.
    matches = [p for p in artifact_paths if p.endswith(RAGAS_TABLE_ARTIFACT)]
    if not matches:
        return None, None

    artifact_path = matches[0]
    local_path = client.download_artifacts(run_id, artifact_path)

    with open(local_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df = normalize_logged_table(payload)
    return df, artifact_path

In [4]:
# Discover runs
if EXPERIMENT_NAME:
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        raise ValueError(f"Experiment not found: {EXPERIMENT_NAME}")
    experiment_ids = [exp.experiment_id]
else:
    experiment_ids = [e.experiment_id for e in client.search_experiments()]

runs_df = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["attribute.start_time DESC"],
    max_results=MAX_RUNS,
)

if runs_df.empty:
    raise ValueError("No MLflow runs found for the current filter.")

print(f"Found {len(runs_df)} runs")
runs_df[["run_id", "experiment_id", "start_time", "status"]].head(10)

Found 4 runs


,run_id,experiment_id,start_time,status
0,47e68305ee86432698b1b1dfa61b78aa,18,2026-04-16 20:22:32.732000+00:00,FINISHED
1,22a0f185e05a49219daef02b5af7225e,18,2026-04-16 19:46:39.137000+00:00,FINISHED
2,4176e466bcd44d6f8a53844ad96ff743,18,2026-04-16 17:32:32.328000+00:00,FINISHED
3,39983b9fddcd433b919cded717671553,18,2026-04-16 17:28:25.503000+00:00,FINISHED


In [5]:
# Pull ragas_scores.json from each run
tables = []
misses = []

for run_id in runs_df["run_id"].tolist():
    df, artifact_path = load_ragas_table_for_run(run_id)
    if df is None:
        misses.append(run_id)
        continue

    df = df.copy()
    df["run_id"] = run_id
    df["artifact_path"] = artifact_path
    tables.append(df)

if not tables:
    raise ValueError("No ragas_scores.json artifacts found in discovered runs.")

all_scores = pd.concat(tables, ignore_index=True)

for col in METRIC_COLS:
    if col in all_scores.columns:
        all_scores[col] = pd.to_numeric(all_scores[col], errors="coerce")

print(f"Runs with ragas_scores.json: {all_scores['run_id'].nunique()}")
print(f"Runs without ragas_scores.json: {len(misses)}")
all_scores.head()

/tmp/ipykernel_132411/3114235660.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_scores = pd.concat(tables, ignore_index=True)


Runs with ragas_scores.json: 4
Runs without ragas_scores.json: 0


,user_input,response,reference,retrieved_contexts,retrieved_context,subdomain,question_class,context_precision,context_recall,factual_correctness,metric_errors,run_id,artifact_path
0,What does the acronym NISQ stand for and what ...,The acronym NISQ stands for Noisy Intermediate...,NISQ stands for Noisy Intermediate-Scale Quant...,[## I. INTRODUCTION \nQuantum software engine...,## I. INTRODUCTION \nQuantum software enginee...,nisq_constraints_and_qsd_constraints,fact_single,0.755556,1.000000,0.67,{},47e68305ee86432698b1b1dfa61b78aa,ragas_scores.json
1,How does the focus of data collection shift as...,As a quantum program progresses through its so...,"In the early stages of development, quantum pr...",[## VII. CONCLUSIONS \nIn this article we dis...,## VII. CONCLUSIONS \nIn this article we disc...,nisq_constraints_and_qsd_constraints,summary,0.542857,1.000000,0.80,{},47e68305ee86432698b1b1dfa61b78aa,ragas_scores.json
2,Based on the constraints of near-term quantum ...,The number of qubits isn't the only thing we c...,There is a strict ceiling on circuit size beca...,[## 4.2 Qubit “quality” \nI’ve emphasized the...,## 4.2 Qubit “quality” \nI’ve emphasized the ...,nisq_constraints_and_qsd_constraints,reasoning,0.666667,0.666667,0.63,{},47e68305ee86432698b1b1dfa61b78aa,ragas_scores.json
3,What is the most popular framework for develop...,I don't know. The context provided discusses e...,The provided documents do not contain sufficie...,[# Toolchain for experiment tracking in iterat...,# Toolchain for experiment tracking in iterati...,nisq_constraints_and_qsd_constraints,unanswerable,0.166667,0.000000,0.00,{},47e68305ee86432698b1b1dfa61b78aa,ragas_scores.json
4,What are the three types of data that can be l...,The three types of data that can be logged per...,"Parameters (key-value pairs), metrics (quantit...","[## Types of System Metrics \nBy default, MLf...","## Types of System Metrics \nBy default, MLfl...",experiment_tracking_fundamentals,fact_single,0.366667,1.000000,0.00,{},47e68305ee86432698b1b1dfa61b78aa,ragas_scores.json


In [6]:
# Select target rows for aggregation
if RUN_SELECTION == "latest":
    latest_run_id = runs_df.iloc[0]["run_id"]
    target = all_scores[all_scores["run_id"] == latest_run_id].copy()
    print(f"Using latest run: {latest_run_id}")
elif RUN_SELECTION == "all":
    target = all_scores.copy()
    print("Using all runs with ragas_scores.json")
else:
    target = all_scores[all_scores["run_id"] == RUN_SELECTION].copy()
    if target.empty:
        raise ValueError(f"No rows found for run_id={RUN_SELECTION}")
    print(f"Using selected run: {RUN_SELECTION}")

required_cols = ["question_class", "subdomain"]
for c in required_cols:
    if c not in target.columns:
        raise ValueError(f"Missing required column in ragas table: {c}")

target[["run_id", "user_input", "question_class", "subdomain"] + [c for c in METRIC_COLS if c in target.columns]].head()

Using latest run: 47e68305ee86432698b1b1dfa61b78aa


,run_id,user_input,question_class,subdomain,context_precision,context_recall,factual_correctness
0,47e68305ee86432698b1b1dfa61b78aa,What does the acronym NISQ stand for and what ...,fact_single,nisq_constraints_and_qsd_constraints,0.755556,1.000000,0.67
1,47e68305ee86432698b1b1dfa61b78aa,How does the focus of data collection shift as...,summary,nisq_constraints_and_qsd_constraints,0.542857,1.000000,0.80
2,47e68305ee86432698b1b1dfa61b78aa,Based on the constraints of near-term quantum ...,reasoning,nisq_constraints_and_qsd_constraints,0.666667,0.666667,0.63
3,47e68305ee86432698b1b1dfa61b78aa,What is the most popular framework for develop...,unanswerable,nisq_constraints_and_qsd_constraints,0.166667,0.000000,0.00
4,47e68305ee86432698b1b1dfa61b78aa,What are the three types of data that can be l...,fact_single,experiment_tracking_fundamentals,0.366667,1.000000,0.00


In [7]:
metric_cols_present = [c for c in METRIC_COLS if c in target.columns]

avg_by_question_class = (
    target.groupby("question_class", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_subdomain = (
    target.groupby("subdomain", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_qclass_and_subdomain = (
    target.groupby(["question_class", "subdomain"], dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

print("Average by question_class")
display(avg_by_question_class)

print("Average by subdomain")
display(avg_by_subdomain)

print("Average by (question_class, subdomain)")
display(avg_by_qclass_and_subdomain)

Average by question_class


,context_precision,context_recall,factual_correctness
question_class,,,
fact_single,0.564444,0.600000,0.268
reasoning,0.617143,0.803333,0.544
summary,0.705238,0.400000,0.378
unanswerable,0.277778,0.200000,0.146


Average by subdomain


,context_precision,context_recall,factual_correctness
subdomain,,,
experiment_tracking_fundamentals,0.613889,0.750000,0.2400
mlflow_tracking_api,0.660714,0.150000,0.5275
nisq_constraints_and_qsd_constraints,0.532937,0.666667,0.5250
qiskit-specific_experiment_tracking_using_mlflow_and_qprov,0.375000,0.437500,0.2550
qprov_provenance_taxonomy,0.523214,0.500000,0.1225


Average by (question_class, subdomain)


context_precision  context_recall  factual_correctness
question_class subdomain                                                                                                 
fact_single    experiment_tracking_fundamentals                             0.366667        1.000000                 0.00
               mlflow_tracking_api                                          1.000000        0.000000                 0.67
               nisq_constraints_and_qsd_constraints                         0.755556        1.000000                 0.67
               qiskit-specific_experiment_tracking_using_mlflo...           0.000000        0.000000                 0.00
               qprov_provenance_taxonomy                                    0.700000        1.000000                 0.00
reasoning      experiment_tracking_fundamentals                             0.633333        1.000000                 0.36
               mlflow_tracking_api                                          0.642857        0.600000                 0.71
               nisq_constraints_and_qsd_constraints                         0.666667        0.666667                 0.63
               qiskit-specific_experiment_tracking_using_mlflo...           0.500000        0.750000                 0.62
               qprov_provenance_taxonomy                                    0.642857        1.000000                 0.40
summary        experiment_tracking_fundamentals                             0.733333        1.000000                 0.60
               mlflow_tracking_api                                          1.000000        0.000000                 0.40
               nisq_constraints_and_qsd_constraints                         0.542857        1.000000                 0.80
               qiskit-specific_experiment_tracking_using_mlflo...           1.000000        0.000000                 0.00
               qprov_provenance_taxonomy                                    0.250000        0.000000                 0.09
unanswerable   experiment_tracking_fundamentals                             0.722222        0.000000                 0.00
               mlflow_tracking_api                                          0.000000        0.000000                 0.33
               nisq_constraints_and_qsd_constraints                         0.166667        0.000000                 0.00
               qiskit-specific_experiment_tracking_using_mlflo...           0.000000        1.000000                 0.40
               qprov_provenance_taxonomy                                    0.500000        0.000000                 0.00